# 07 · The simulator (phase 7)

Produced by `make sim` → `results/sim/`. This notebook only reads those files.

Two things are worth your attention before any policy comparison, and both are on this page:

1. **The engine agrees with a closed form.** Stripped to one agency with no incidents, the
   simulator *is* an M/M/c queue, so Erlang-C solves exactly the same system. That is the only
   check here that compares the simulator against an answer it did not produce itself.
2. **The cameras see almost nothing.** Of the incidents that happen, 0.3 % are detected by a
   camera — which bounds what *any* CCTV-triggered policy can achieve, before a single
   dispatch decision is made (D-032 item 1).

Policy comparisons are deliberately **not** here: a week holds only ~36 incidents and ~2 reach a
full response, so one replication is noise. That is phase 10's job, with 30 replications and
common random numbers.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path('..') / 'results' / 'sim'
funnel = json.loads((RESULTS / 'evidence_funnel.json').read_text(encoding='utf-8'))
queue = json.loads((RESULTS / 'queue_validation.json').read_text(encoding='utf-8'))
summaries = {
    path.stem.replace('summary_normal_', '').replace('_synthetic', ''):
        json.loads(path.read_text(encoding='utf-8'))
    for path in sorted(RESULTS.glob('summary_normal_*_synthetic.json'))
}
print('policies with a saved run:', sorted(summaries))
print('funnel weeks:', funnel['replications'], '| queue validation passed:', queue['passed'])

## 1. Does the engine queue correctly?

The reduction is exact rather than approximate: Poisson arrivals, exponential service, $c$
identical servers, one FIFO queue, no abandonment, no pre-emption. So a disagreement beyond
sampling error is a bug, not a modelling difference.

The light-load cases could agree by luck — nothing waits. The case that carries the check is
$\rho = 0.85$, where about a fifth of jobs queue.

In [ ]:
rows = []
for case in queue['cases']:
    rows.append({
        'rho': case['target_utilisation'],
        'jobs': case['simulated']['jobs_arrived'],
        'util (sim)': case['simulated']['utilisation'],
        'util (Erlang-C)': case['analytic']['utilisation'],
        'P(wait) sim': case['simulated']['p_wait'],
        'P(wait) Erlang-C': case['analytic']['p_wait'],
        'Wq sim (s)': case['simulated']['expected_wait_s'],
        'Wq Erlang-C (s)': case['analytic']['expected_wait_s'],
    })
table = pd.DataFrame(rows).set_index('rho')
print('worst absolute difference across all loads:')
for key, value in queue['worst_absolute_difference'].items():
    print(f'  {key:18s} {value:.4f}')
table.round(4)

The agreement is tight where it matters. What this validates is the **clock, the queue
discipline and the busy-time accounting** — the things every other number rests on. It does
**not** validate travel, on-scene time or the policies, none of which has a closed form, and the
PASS must not be read as more than that.

## 2. What the cameras actually see

Each stage below is bounded by a quantity an earlier phase measured, so this figure is also a
consistency check on phases 2, 3b and 6. The dashed markers are what those phases predict.

In [ ]:
# Palette: reference categorical slots (validated light-mode set), text and grid neutrals.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#6250d6']
SURFACE, TEXT, TEXT2, MUTED, GRID, AXIS = ('#fcfcfb', '#0b0b0b', '#52514e', '#898781',
                                           '#e1e0d9', '#c3c2b7')
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': AXIS, 'axes.labelcolor': TEXT2,
                     'xtick.color': MUTED, 'ytick.color': MUTED,
                     'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE})

totals = funnel['totals']
stages = ['incidents', 'within_camera_range', 'detected_by_a_camera']
labels = ['happened', 'in camera range', 'detected by a camera']
values = [totals[s] for s in stages]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={'width_ratios': [1.15, 1]})
ax = axes[0]
bars = ax.barh(range(len(values))[::-1], values, color=SERIES[0], height=0.6)
bars[1].set_color(SERIES[3])
bars[2].set_color(SERIES[1])
for y, value in enumerate(values):
    share = value / values[0] if values[0] else 0
    ax.annotate(f'{value}  ({share:.1%})', (value, len(values) - 1 - y), xytext=(6, 0),
                textcoords='offset points', va='center', color=TEXT2, fontsize=9)
ax.set_yticks(range(len(values))[::-1])
ax.set_yticklabels(labels, color=TEXT2)
ax.set_xlim(0, values[0] * 1.45)
ax.set_xlabel(f"incidents over {funnel['weeks']} simulated weeks")
ax.set_title('Almost nothing reaches a camera', loc='left', color=TEXT)

# measured share vs what phases 2/3b/6 predict
ax = axes[1]
shares = funnel['shares']
expected = funnel['expected_from_earlier_phases']
names = ['in range\n(of all incidents)', 'detected\n(of those in range)']
got = [shares['within_camera_range'], shares['detected_of_in_range']]
want = [expected['within_camera_range'], expected['detected_of_in_range']]
x = range(len(names))
ax.bar(x, got, width=0.45, color=SERIES[0], label='measured here')
for i, value in enumerate(want):
    ax.plot([i - 0.33, i + 0.33], [value, value], color=SERIES[1], lw=2, ls='--')
ax.plot([], [], color=SERIES[1], lw=2, ls='--', label='predicted by D-031 / D-029')
ax.set_xticks(list(x))
ax.set_xticklabels(names)
ax.set_ylabel('share')
ax.legend(frameon=False, fontsize=8)
ax.set_title('Consistent with the earlier phases', loc='left', color=TEXT)
for a in axes:
    a.grid(axis='x' if a is axes[0] else 'y', color=GRID, lw=1)
    a.set_axisbelow(True)
    for side in ('top', 'right'):
        a.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

print(f"alarms raised: {totals['alarms']:,} of which {totals['true_alarms']} true "
      f"({shares['alarm_precision']:.7%})")
print(f"incidents with a 112 call: {shares['incidents_with_a_call']:.1%}")

Read the right-hand panel as the reconciliation: the share of incidents falling in camera
range matches D-031's 3.95 % risk-weighted coverage, and the share of those actually detected
matches D-029's 0.306 effective detection. Both sit below their predicted value here, which is
what a handful of events per 8 weeks does to a Poisson count — **not** evidence of a
discrepancy.

The consequence is the important part. **0.3 % of incidents are camera-detected**, and the
operators would see about 1.5 million alarms to find one real event. A CCTV-triggered policy
therefore has essentially nothing to trigger on at this camera deployment, so it can differ from
the call-based baseline only through the *cost* of chasing false alarms. Phase 10's decisive
sweep is camera coverage; the joint-dispatch question is separable and needs no camera.

## 3. A week of the city, per policy

One replication each, which is the phase-7 acceptance run and **not** a comparison. What it shows
is that the lifecycle closes: units become busy, go to hospital, and come back.

Utilisation is the number to look at. Before the background workload of D-031 item 3 the
simulated fleet ran at 0.3 %; a city at 0.3 % utilisation has no queues, so joint dispatch would
have beaten sequential dispatch for a reason unrelated to coordination.

In [ ]:
rows = []
for policy, summary in summaries.items():
    util = summary['utilisation_after_warmup'][0]
    hosp = summary['hospitals'][0] if summary.get('hospitals') else {}
    rows.append({
        'policy': policy,
        'log rows': summary['rows'],
        'wall s/week': round(summary['wall_seconds'][0], 1),
        'util ambulance': round(util['ambulance'], 3),
        'util police': round(util['police'], 3),
        'util fire': round(util['fire'], 3),
        'handovers': hosp.get('handovers'),
        'no capable hospital': hosp.get('patients_with_no_capable_hospital'),
    })
runs = pd.DataFrame(rows).set_index('policy')
print('evidence digests identical across policies (common random numbers):',
      len({s['evidence_digests'][0] for s in summaries.values()}) == 1)
runs

The digest line is the check that matters for every later comparison: all policies ran on
the **identical** week, so a phase-10 difference between them is the policy rather than the
weather.

## 4. What is not on this page

- **Policy comparisons.** ~2 incidents per week reach a full response, so a single replication
  cannot separate B0 from FULL. Phase 10 does this with 30 replications.
- **Expected survival.** It is computed but stays a secondary outcome: no defensible per-minute
  survival-vs-time curve for road trauma could be confirmed at the source, so every curve in
  `configs/survival.yaml` is ILLUSTRATIVE and a validator refuses to make it primary (D-032
  item 5).
- **Anything about real Pune response times.** The incidents, the detector outcomes and the calls
  are all synthetic. What the simulator measures is how a dispatch policy behaves on this
  generator, which is what the research question needs.